# AHA-Fed — statistical power run (10 seeds)

The paper's comparisons currently use 3 seeds. A Wilcoxon signed-rank test on
n=3 cannot return p below 0.25 no matter how large the effect, so no claim can
be called significant. This notebook re-runs the core comparisons at **10 seeds**,
which brings the attainable p down to ~0.002.

**Settings:** Accelerator → GPU T4 x2, Internet → On, Input → `aha-fed-code`.

Roughly 3–4 h. Runs are skipped when their JSON already exists, so a timeout resumes.

In [ ]:
import glob, os, shutil, sys, zipfile

REPO = '/kaggle/working/repo'

def find_source():
    for hit in glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True):
        return os.path.dirname(os.path.dirname(hit)), 'tree'
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        return zips[0], 'zip'
    raise SystemExit('Attach the aha-fed-code dataset (right panel -> Add Input).')

if not os.path.isdir(os.path.join(REPO, 'paper')):
    src, kind = find_source()
    os.makedirs(REPO, exist_ok=True)
    if kind == 'zip':
        with zipfile.ZipFile(src) as z:
            z.extractall(REPO)
    else:
        shutil.copytree(src, REPO, dirs_exist_ok=True)

os.chdir(REPO)
sys.path.insert(0, REPO)
assert os.path.isdir('paper')
print('cwd:', os.getcwd())

In [ ]:
!pip install -q medmnist opacus
# torch is deliberately NOT imported here — the next cell may need to
# replace the wheel, which is impossible once torch is loaded.


In [ ]:
# Kaggle's torch 2.10 is built for sm_70+ and cannot run on the P100 (sm_60)
# that sessions are sometimes assigned. Probe with a subprocess so the check
# does not pin a torch version into this kernel, then install a compatible
# build before anything imports torch here.
import subprocess, sys

PROBE = (
    'import torch;'
    'a=torch.randn(4,3,8,8,device="cuda");'
    'w=torch.randn(2,3,3,3,device="cuda");'
    'torch.nn.functional.conv2d(a,w).sum().item();'
    'print(torch.cuda.get_device_name(0), torch.__version__)'
)
r = subprocess.run([sys.executable, '-c', PROBE],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])

if r.returncode != 0:
    print('GPU cannot run the installed torch; installing cu118 build ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.4.1', 'torchvision==0.19.1',
                    '--index-url', 'https://download.pytorch.org/whl/cu118'],
                   check=True)
    r2 = subprocess.run([sys.executable, '-c', PROBE],
                        capture_output=True, text=True)
    print(r2.stdout.strip() or r2.stderr.strip()[-400:])
    if r2.returncode != 0:
        raise SystemExit('no working torch/GPU combination on this session')
    print('cu118 build works on this GPU')
else:
    print('default torch works on this GPU')


## PathMNIST — 5 aggregators × 10 seeds

The paper's primary comparison. Seeds 42/123/456 already exist and are skipped.

In [ ]:
# The PathMNIST and PneumoniaMNIST matrices are already complete and were
# downloaded; this session only runs the label-noise sweep below.
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS10 = [42, 123, 456, 7, 13, 21, 99, 2024, 31337, 555]
print('imports ready; baseline matrices skipped')


## PneumoniaMNIST (chest radiography) — 5 aggregators × 10 seeds

In [ ]:
print('PneumoniaMNIST matrix already complete — skipped')


## Label-noise sweep — 10 seeds

This is where the paper's robustness claim lives, so it needs the most power.

In [ ]:
# Noise sweep is complete and downloaded. This session raises the
# heterogeneity and client-scaling sweeps from 3 seeds to 10, since the
# heterogeneity result is the paper's headline claim.
PATH = {'dataset': 'pathmnist', 'num_classes': 9, 'samples_per_client': 1000}

for h in [0.0, 0.3, 0.5, 0.7, 0.9]:
    for s in SEEDS10:
        suffix = '' if s == 42 else f'_s{s}'
        run_one(OUT_DIR / f'real_het_fedavg_h{h}{suffix}.json',
                aggregator='fedavg', heterogeneity=h, seed=s, **PATH)
        run_one(OUT_DIR / f'real_het_aha_h{h}{suffix}.json',
                aggregator='adaptive', alpha=1.0, beta=4.0, gamma=4.0,
                heterogeneity=h, seed=s, **PATH)
print('heterogeneity sweep done')

for nc in [4, 8, 12]:
    spc = max(200, 4000 // nc)
    common = {'dataset': 'pathmnist', 'num_classes': 9, 'num_clients': nc,
              'samples_per_client': spc, 'noisy_client': nc - 1}
    for s in SEEDS10:
        suffix = '' if s == 42 else f'_s{s}'
        run_one(OUT_DIR / f'real_scale_fedavg_k{nc}{suffix}.json',
                aggregator='fedavg', seed=s, **common)
        run_one(OUT_DIR / f'real_scale_aha_k{nc}{suffix}.json',
                aggregator='adaptive', alpha=1.0, beta=4.0, gamma=4.0,
                seed=s, **common)
print('scaling sweep done')


## Summary and export

In [ ]:
!python paper/summarize_real_results.py

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/aha_fed_seeds.zip runs/paper/real
!ls -lh /kaggle/working/aha_fed_seeds.zip